In [ ]:
#!pip install openai

In [4]:
import json
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI

In [10]:
OPENAI_MODEL = "gpt-4o-mini" #gpt-5-mini
TEMPERATURE = 0

In [27]:
SYSTEM_PROMPT = (
    "Sei un estrattore di informazioni. "
    "Devi rispondere SOLO in formato json valido (oggetto JSON), senza testo extra, "
    "senza markdown e senza code fences. "
    "La risposta deve essere un unico oggetto json."
)

USER_INSTRUCTIONS = """Estrai l'elenco strutturato dei centri/sportelli/case citati nel testo qui sotto.
Regole:
- 'tipo' ∈ {Centro Antiviolenza, Sportello collegato, Casa Rifugio, Altro}
- Compila comuni/indirizzi solo se esplicitamente presenti.
- Indica in 'ente_capofila' se dal testo emerge (es. “Comune di Bra”).
- In 'note' aggiungi contesto utile (es. “sportelli decentrati del CAV n.10/A del Cuneese”, “collegamento al 1522”, “nuovo centro”).
Testo:
"""

In [28]:
testo_protocollo = """PROTOCOLLO TERRITORIALE TRA COMUNE DI BRA, IL COMUNE DI ALBA, IL CONSORZIO
SOCIO ASSISTENZIALE ALBA LANGHE ROERO, L’A.S.L CN 2 E L;ASSOCIAZIONE MAI+SOLE,
PER LA REALIZZAZIONE DI UN NUOVO CENTRO ANTIVIOLENZA

TRA

- Il Comune di Bra (Ente capofila), in qualita di Ente Locale territoriale e soggetto gestore dei Servizi Socio
Assistenziali, con sede legale a Bra, piazza Caduti per la Liberta n. 14, C.F. n. 82000150043/ P.Iva n.
00493130041, rappresentato dal Sindaco Giovanni Fogliato nato a Bra il 24/09/1961 domiciliato, ai fini del
presente accordo, presso la sede legale dell’ Ente;

E

- Il Consorzio Socio Assistenziale Alba Langhe Roero, con sede legale ad Alba (CN), Via A. Diaz, n. 8, CF/
P. IVA: n. 02797980048, legalmente rappresentato dalla dott.ssa Loredana Defilippi n. ad Alba il
23/06/1960 , domiciliata ai fini del presente accordo presso la sede legale dell’ Ente;

E

- Il Comune di Alba, con sede legale ad Alba in Piazza Risorgimento n. 1 C.F/P.IVA 00184260040,
rappresentato dal sindaco pro tempore CARLO BO, nato Carmagnola (TO) il 15/08/1970, domicliato, ai
fine del presente accordo presso la sede legale dell'Ente;

E

LASL CN2 , con sede legale ad Alba (CN) Via Vida n. 10, CF/PIVA n. 02419170044, rappresentata dal
legale rappresentante Massimo VEGLIO nato Torino il 18/07/1959, domiciliato ai fini del presente accordo
presso la sede legale dell’ Ente;

E

- L’Associazione MAI+SOLE, con — sede legale in Savigliano (CN), P. IVA/C.F: n. 95019460047
rappresentata dal legale rappresentante: FIORITO Adonella, nata a Villafalletto (CN) il 20/09/1956,
domiciliata ai fini del presente accordo a Savigliano in Via Teatro n. 2;

RICHIAMATI

e la Legge 27 giugno 2013 n.77 “Ratifica ed esecuzione della Convenzione del Consiglio d’Europa sulla
prevenzione e la lotta contro la violenza nei confronti delle donne e la violenza domestica, fatta ad
Istanbul I’11 maggio 2011”;

e la Legge 15 ottobre 2013, n. 119, “Conversione in legge, con modificazioni, del decreto-legge 14 agosto
2013, n. 93, recante disposizioni urgenti in materia di sicurezza e per il contrasto della violenza di
genere, nonché in tema di protezione civile e di commissariamento delle province”, che individua, tra gli
obiettivi di cui all’art. 5, comma 2, “d) potenziare le forme di assistenza e di sostegno alle donne vittime
di violenza e ai loro figli attraverso modalita omogenee di rafforzamento della rete dei servizi
territoriali, dei centri antiviolenza e dei servizi di assistenza alle donne vittime di violenza”;

e ’Intesa CU n. 146 del 27 novembre 2014, tra il Governo e le Regioni, le Province autonome di Trento e
di Bolzano e le Autonomie locali, relativa ai requisiti minimi dei Centri antiviolenza e delle Case
Rifugio;

e la Legge regionale 18 marzo 2009, n. 8, “Integrazione delle politiche di pari opportunita di genere nella
Regione Piemonte e disposizioni per l'istituzione dei bilanci di genere”, che all’articolo 2, comma h)
recita: “promuovere e sostenere azioni volte a prevenire la violenza fondata sul genere e la tratta delle
donne, anche attivando piani e programmi per la tutela delle vittime’’;

1

Comune di Bra
Il Sindaco

Comune di Alba
II Sindaco

Consorzio Socio Assistenziale Alba Langhe Roero
Il Presidente Dr.ssa Loredana Defilippi

ASL 
Il Direttore Generale 

Associazione MAI+SOLE
Il legale Rappresentante 
"""

In [29]:
# Chiave API
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [30]:
resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    temperature=TEMPERATURE,
    response_format={
        "type": "json_object"
    },
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_INSTRUCTIONS + testo_protocollo}
    ],
)

# Con json_schema, la risposta è già JSON
data = json.loads(resp.choices[0].message.content)
print(json.dumps(data, indent=2, ensure_ascii=False))


{
  "centri": [
    {
      "tipo": "Centro Antiviolenza",
      "ente_capofila": "Comune di Bra",
      "comune": "Bra",
      "indirizzo": "Piazza Caduti per la Libertà n. 14",
      "note": "Realizzazione di un nuovo centro antiviolenza"
    },
    {
      "tipo": "Altro",
      "ente_capofila": "Consorzio Socio Assistenziale Alba Langhe Roero",
      "comune": "Alba",
      "indirizzo": "Via A. Diaz, n. 8",
      "note": "Partecipazione al protocollo territoriale"
    },
    {
      "tipo": "Altro",
      "ente_capofila": "Comune di Alba",
      "comune": "Alba",
      "indirizzo": "Piazza Risorgimento n. 1",
      "note": "Partecipazione al protocollo territoriale"
    },
    {
      "tipo": "Altro",
      "ente_capofila": "ASL CN2",
      "comune": "Alba",
      "indirizzo": "Via Vida n. 10",
      "note": "Partecipazione al protocollo territoriale"
    },
    {
      "tipo": "Altro",
      "ente_capofila": "Associazione MAI+SOLE",
      "comune": "Savigliano",
      "indirizzo":